In [1]:

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.layers import Input, Dense, Concatenate
from tensorflow.keras.models import Model

from sklearn.model_selection import train_test_split



In [2]:
np.random.seed(42)

n = 1000

hours_studied = np.random.randint(1,10,n)
attendance = np.random.randint(50,100,n)
previous_marks = np.random.randint(30,100,n)

age = np.random.randint(16,22,n)
sleep_hours = np.random.randint(4,10,n)
exercise = np.random.randint(0,3,n)

In [3]:
exam_score = (
    hours_studied*5
    + attendance*0.3
    + previous_marks*0.4
    + np.random.normal(0,5,n)
)

In [4]:
pass_fail = (exam_score >= 60).astype(int)

In [5]:
df = pd.DataFrame({
    'hours_studied':hours_studied,
    'attendance':attendance,
    'previous_marks':previous_marks,
    'age':age,
    'sleep_hours':sleep_hours,
    'exercise':exercise,
    'exam_score':exam_score,
    'pass_fail':pass_fail
})

df.head()

,hours_studied,attendance,previous_marks,age,sleep_hours,exercise,exam_score,pass_fail
0,7,73,75,16,4,1,83.314919,1
1,4,72,72,16,6,1,63.977079,1
2,8,57,50,19,5,1,81.989078,1
3,5,96,52,18,7,1,64.071765,1
4,7,86,76,20,5,0,96.967952,1


In [6]:
X1 = df[['hours_studied',
         'attendance',
         'previous_marks']]

In [7]:
X2 = df[['age',
         'sleep_hours',
         'exercise']]

In [8]:
y_score = df['exam_score']

y_pass = df['pass_fail']

In [9]:
X1_train, X1_test, X2_train, X2_test, y_score_train, y_score_test, y_pass_train, y_pass_test = train_test_split(
    X1,
    X2,
    y_score,
    y_pass,

    test_size=0.2,
    random_state=42
)

In [10]:
academic_input = Input(shape=(3,),
                       name='academic_input')

x1 = Dense(32,
           activation='relu')(academic_input)

x1 = Dense(16,
           activation='relu')(x1)

In [11]:
personal_input = Input(shape=(3,),
                       name='personal_input')

x2 = Dense(32,
           activation='relu')(personal_input)

x2 = Dense(16,
           activation='relu')(x2)

In [12]:
merged = Concatenate()([x1,x2])

In [13]:
shared = Dense(32,
               activation='relu')(merged)

shared = Dense(16,
               activation='relu')(shared)

In [14]:
score_output = Dense(
    1,
    activation='linear',
    name='score_output'
)(shared)

In [15]:
pass_output = Dense(
    1,
    activation='sigmoid',
    name='pass_output'
)(shared)

In [16]:
model = Model(
    inputs=[academic_input,
            personal_input],

    outputs=[score_output,
             pass_output]
)

In [17]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ academic_input      │ (None, 3)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ personal_input      │ (None, 3)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │        128 │ academic_input[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 32)        │        128 │ personal_input[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 16)        │        528 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 16)        │        528 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 32)        │          0 │ dense_1[0][0],    │
│ (Concatenate)       │                   │            │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 32)        │      1,056 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 16)        │        528 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ score_output        │ (None, 1)         │         17 │ dense_5[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pass_output (Dense) │ (None, 1)         │         17 │ dense_5[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,930 (11.45 KB)

 Trainable params: 2,930 (11.45 KB)

 Non-trainable params: 0 (0.00 B)

In [18]:
model.compile(
    optimizer='adam',

    loss={
        'score_output':'mse',
        'pass_output':'binary_crossentropy'
    },

    metrics={
        'score_output':['mae'],
        'pass_output':['accuracy']
    }
)

In [19]:
model.compile(
    optimizer='adam',

    loss={
        'score_output':'mse',
        'pass_output':'binary_crossentropy'
    },

    metrics={
        'score_output':['mae'],
        'pass_output':['accuracy']
    }
)

In [20]:
history = model.fit(
    [X1_train,X2_train],

    [y_score_train,
     y_pass_train],

    epochs=30,
    batch_size=32,

    validation_split=0.2
)

Epoch 1/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - loss: 3379.4133 - pass_output_accuracy: 0.7563 - pass_output_loss: 0.9994 - score_output_loss: 3378.4141 - score_output_mae: 55.4494 - val_loss: 1712.3043 - val_pass_output_accuracy: 0.7188 - val_pass_output_loss: 1.2748 - val_score_output_loss: 1711.0295 - val_score_output_mae: 38.6541
Epoch 2/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 659.0585 - pass_output_accuracy: 0.7563 - pass_output_loss: 1.3344 - score_output_loss: 657.7241 - score_output_mae: 21.2209 - val_loss: 260.0157 - val_pass_output_accuracy: 0.7188 - val_pass_output_loss: 1.3312 - val_score_output_loss: 258.6845 - val_score_output_mae: 13.6634
Epoch 3/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 234.5196 - pass_output_accuracy: 0.7312 - pass_output_loss: 0.9120 - score_output_loss: 233.6075 - score_output_mae: 12.5569 - val_loss: 189.0455 - val_pass_output_accuracy: 0.5688 - val_pass_output_loss: 0.6775 - val_score_output_loss: 188.3680 - val_score_output_

In [21]:
model.evaluate(
    [X1_test,X2_test],

    [y_score_test,
     y_pass_test]
)

7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 24.4609 - pass_output_accuracy: 0.8750 - pass_output_loss: 0.3057 - score_output_loss: 22.8873 - score_output_mae: 3.9574 


[24.46087074279785,
 22.887327194213867,
 0.30570751428604126,
 0.875,
 3.9573519229888916]

In [22]:
prediction = model.predict(
    [
        np.array([[8,90,85]]),
        np.array([[18,8,1]])
    ]
)

print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step
[array([[99.939735]], dtype=float32), array([[0.9777175]], dtype=float32)]
